In [ ]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from collections import Counter

In [ ]:
df = pd.read_csv("../data/raw/IMDB Dataset.csv")

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.drop_duplicates(inplace = True)

In [ ]:
df.shape

In [ ]:
df["sentiment"].value_counts()

In [ ]:
# Check average and max review word count (helps to pick max_len for RNN later)
df["review"].str.split().str.len().describe()

## Text - preprocessing

### 1.Convering to lower case

In [ ]:
df["review"] = df["review"].str.lower()

### 2.Removing the URL's

In [ ]:
def remove_urls(text):
    return re.sub(r"https?://\S+|\bwww\.\S+", " ", text)

df["review"] = df["review"].apply(remove_urls)

### 3.removing html tags

In [ ]:
def remove_html_tags(text):
    return re.sub(r"<.*?>", " ", text).strip()

df["review"] = df["review"].apply(remove_html_tags)

### 4.removing punctuations

In [ ]:
def remove_punctuations(text):
    return re.sub(r"[^a-z0-9]+", " ", text).strip()

df["review"] = df["review"].apply(remove_punctuations)

### 5.removing stopwords

In [ ]:
# 1. Load ONCE outside the function as a set for O(1) lookups
stop_words = set(stopwords.words("english")) - {"no", "nor", "not"}

def remove_stopwords(text):
    tokens = word_tokenize(text)  
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(filtered_tokens)

df["review"] = df["review"].apply(remove_stopwords)

### 6.stemming

In [ ]:
# converting the words into there base forms such as running=> run 
# we use porter stemming

ps = PorterStemmer()
def stemming(text) :
    tokens = text.split()

    stemmed_words = [ps.stem(token) for token in tokens]

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df

### 7.Encoding

In [ ]:
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [ ]:
y = df["sentiment"]
print(y)

In [ ]:
print(dict(enumerate(le.classes_)))

### 8.Vectorization

In [ ]:
# applying sequence vectorization

# 80% Train, 10% Validation, 10% Test
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    df["review"], y, test_size=0.2, random_state=42, stratify=y
)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(len(X_train_text), len(X_val_text), len(X_test_text))

In [ ]:
word_counts = Counter()
for review in X_train_text:
    word_counts.update(review.split())

MAX_VOCAB_SIZE = 10000  # Keep top 10,000 tokens (including <PAD> and <UNK>)
word2idx = {"<PAD>": 0, "<UNK>": 1}

for word, freq in word_counts.most_common(MAX_VOCAB_SIZE - 2):
    word2idx[word] = len(word2idx)

print(f"Total unique words in train: {len(word_counts)}")
print(f"Vocab size kept: {len(word2idx)}")

In [ ]:
list(word2idx.items())[:10]

In [ ]:
import numpy as np

lengths = X_train_text.apply(lambda x: len(x.split()))
print("Median (50th):", int(np.percentile(lengths, 50)))
print("90th percentile:", int(np.percentile(lengths, 90)))
print("95th percentile:", int(np.percentile(lengths, 95)))
print("Max length:", lengths.max())

In [ ]:
MAX_LEN = 300

def encode_and_pad(text, word2idx, max_len):
    words = text.split()
    ids = []

    # Step 1: Turn each word into its integer ID
    for word in words:
        if word in word2idx:
            ids.append(word2idx[word])
        else:
            ids.append(word2idx["<UNK>"])  # Use 1 if word is not in vocab

    # Step 2: If the review is longer than 300, keep only the first 300 IDs
    if len(ids) > max_len:
        ids = ids[:max_len]

    # Step 3: If the review is shorter than 300, keep adding 0 until length hits 300
    while len(ids) < max_len:
        ids.append(word2idx["<PAD>"])      # Use 0 for padding

    return ids


def encode_dataset(reviews, word2idx, max_len):
    encoded_reviews = []
    for review in reviews:
        encoded = encode_and_pad(review, word2idx, max_len)
        encoded_reviews.append(encoded)

    return np.array(encoded_reviews, dtype=np.int64)


# Run on all three splits
X_train = encode_dataset(X_train_text, word2idx, MAX_LEN)
X_val   = encode_dataset(X_val_text, word2idx, MAX_LEN)
X_test  = encode_dataset(X_test_text, word2idx, MAX_LEN)

print("X_train shape:", X_train.shape)  # (39665, 300)
print("X_val shape:  ", X_val.shape)    # (4958, 300)
print("X_test shape: ", X_test.shape)   # (4959, 300)

In [ ]:
print(X_train[0][:20])